# Первичный просмотр и предобработк данных Olist
Кейс 1: «Исследование факторов, влияющих на своевременную доставку и удовлетворённость клиентов».

На этом этапе мы загружаем таблицы, смотрим, что в них есть, ищем пропуски, дубликаты и странные значения, и разбираемся, как таблицы связаны между собой. Глубокий анализ — на следующих этапах.

Данные: [Brazilian E-Commerce Public Dataset by Olist (Kaggle)](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce). CSV-файлы нужно положить в папку `data/`. Таблица геолокации большая, поэтому хранится в архиве `olist_geolocation_dataset.zip` — pandas читает его без распаковки.

In [1]:
import pandas as pd

DATA_DIR = 'data/'
files = {
    'orders':    'olist_orders_dataset.csv',
    'items':     'olist_order_items_dataset.csv',
    'reviews':   'olist_order_reviews_dataset.csv',
    'payments':  'olist_order_payments_dataset.csv',
    'customers': 'olist_customers_dataset.csv',
    'products':  'olist_products_dataset.csv',
    'sellers':   'olist_sellers_dataset.csv',
    'cat_tr':    'product_category_name_translation.csv',
}
df = {name: pd.read_csv(DATA_DIR + f) for name, f in files.items()}
df['geo'] = pd.read_csv(DATA_DIR + 'olist_geolocation_dataset.zip')

## 1. Размер таблиц и столбцы

In [2]:
for name, t in df.items():
    print(f'{name}: {t.shape[0]} строк, {t.shape[1]} столбцов')
    print('   ', list(t.columns))

orders: 99441 строк, 8 столбцов
    ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
items: 112650 строк, 7 столбцов
    ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
reviews: 99224 строк, 7 столбцов
    ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
payments: 103886 строк, 5 столбцов
    ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
customers: 99441 строк, 5 столбцов
    ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
products: 32951 строк, 9 столбцов
    ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'produc

## 2. Пропуски

In [3]:
for name, t in df.items():
    na = t.isna().sum()
    na = na[na > 0]
    if len(na):
        print(f'--- {name}')
        print(na.to_string())

--- orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
--- reviews
review_comment_title      87656
review_comment_message    58247
--- products
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2


## 3. Дубликаты

In [4]:
for name, t in df.items():
    print(f'{name}: полных дубликатов строк — {t.duplicated().sum()}')
print()
print('Заказов с несколькими товарами:', (df['items'].groupby('order_id').size() > 1).sum())
print('Заказов с несколькими оплатами:', (df['payments'].groupby('order_id').size() > 1).sum())
print('Заказов с несколькими отзывами:', (df['reviews'].groupby('order_id').size() > 1).sum())

orders: полных дубликатов строк — 0
items: полных дубликатов строк — 0
reviews: полных дубликатов строк — 0
payments: полных дубликатов строк — 0
customers: полных дубликатов строк — 0
products: полных дубликатов строк — 0
sellers: полных дубликатов строк — 0
cat_tr: полных дубликатов строк — 0
geo: полных дубликатов строк — 261831

Заказов с несколькими товарами: 9803
Заказов с несколькими оплатами: 2961
Заказов с несколькими отзывами: 547


## 4. Статусы заказов и даты
Даты хранятся как текст — переводим в формат даты.

In [5]:
orders = df['orders'].copy()
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
             'order_delivered_customer_date', 'order_estimated_delivery_date']
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

print(orders['order_status'].value_counts().to_string())
print()
print('Период покупок:', orders['order_purchase_timestamp'].min().date(), '—', orders['order_purchase_timestamp'].max().date())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2

Период покупок: 2016-09-04 — 2018-10-17


In [6]:
orders['order_purchase_timestamp'].dt.to_period('M').value_counts().sort_index()

order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M

## 5. Странные значения в датах

In [7]:
dl = orders[orders['order_status'] == 'delivered']
print('Доставленных заказов:', len(dl))
print('Доставлены, но нет даты доставки:', dl['order_delivered_customer_date'].isna().sum())
print('Передача перевозчику раньше покупки:', (dl['order_delivered_carrier_date'] < dl['order_purchase_timestamp']).sum())
print('Доставка покупателю раньше передачи перевозчику:', (dl['order_delivered_customer_date'] < dl['order_delivered_carrier_date']).sum())

Доставленных заказов: 96478
Доставлены, но нет даты доставки: 8
Передача перевозчику раньше покупки: 165
Доставка покупателю раньше передачи перевозчику: 23


## 6. Как связаны таблицы
Главная таблица — заказы (`order_id`). К ней по `order_id` присоединяются товары, оплаты и отзывы, по `customer_id` — покупатели. Товары связаны с карточками товаров (`product_id`) и продавцами (`seller_id`). Геолокация связана с покупателями и продавцами только через префикс почтового индекса (`zip_code_prefix`).

In [8]:
o_ids = set(orders['order_id'])
print('order_id уникален в заказах:', orders['order_id'].is_unique)
print('Товары без заказа:', (~df['items']['order_id'].isin(o_ids)).sum())
print('Отзывы без заказа:', (~df['reviews']['order_id'].isin(o_ids)).sum())
print('Заказы без покупателя:', (~orders['customer_id'].isin(df['customers']['customer_id'])).sum())
print('Товары без продавца:', (~df['items']['seller_id'].isin(df['sellers']['seller_id'])).sum())

order_id уникален в заказах: True
Товары без заказа: 0
Отзывы без заказа: 0
Заказы без покупателя: 0
Товары без продавца: 0


## 7. Таблица геолокации
В таблице координаты (широта и долгота) для префиксов почтовых индексов. Она может пригодиться, чтобы посчитать расстояние между продавцом и покупателем.

In [9]:
geo = df['geo']
print('Строк:', len(geo), '| уникальных индексов:', geo['geolocation_zip_code_prefix'].nunique())
print('Полных дубликатов:', geo.duplicated().sum())
in_brazil = geo['geolocation_lat'].between(-33.8, 5.3) & geo['geolocation_lng'].between(-74, -34.7)
print('Координаты за пределами Бразилии:', (~in_brazil).sum())
zips = set(geo['geolocation_zip_code_prefix'])
print('Покупатели, чьего индекса нет в геолокации:', (~df['customers']['customer_zip_code_prefix'].isin(zips)).sum())
print('Продавцы, чьего индекса нет в геолокации:', (~df['sellers']['seller_zip_code_prefix'].isin(zips)).sum())
print('Примеры написания Сан-Паулу:', list(geo.loc[geo['geolocation_city'].str.contains('paulo') & (geo['geolocation_state'] == 'SP'), 'geolocation_city'].unique()[:4]))

Строк: 1000163 | уникальных индексов: 19015
Полных дубликатов: 261831
Координаты за пределами Бразилии: 42
Покупатели, чьего индекса нет в геолокации: 278
Продавцы, чьего индекса нет в геолокации: 7
Примеры написания Сан-Паулу: ['sao paulo', 'são paulo', 'sãopaulo', 'sa£o paulo']


## Что мы поняли
- Таблицы связываются между собой без потерь.
- В одном заказе может быть несколько товаров, оплат и отзывов — перед объединением их нужно свести к одной строке на заказ, иначе суммы задвоятся.
- Даты нужно переводить из текста в формат даты.
- Для анализа доставки подходят заказы со статусом `delivered` и известной датой доставки.
- В начале (2016) и в конце (сентябрь–октябрь 2018) периода заказов очень мало — эти месяцы, скорее всего, уберём.
- Есть небольшое число заказов со странными датами — нужно решить, удалять ли их.
- У части товаров нет категории.
- В геолокации много дубликатов, есть ошибочные координаты за пределами Бразилии и разное написание городов. На один индекс приходится много точек — если будем считать расстояния, нужно свести их к одной точке на индекс (например, среднее).

**Следующий шаг:** объединить таблицы и создать признаки (опоздание, срок доставки, доставка в другой штат и т. д.). Решить в команде, используем ли расстояние до покупателя.